# Data Cleaning

In [1]:
import os

import pandas as pd

ROOT_PATH = os.path.dirname(os.getcwd())
DATA_FOLDER_PATH = os.path.join(ROOT_PATH, 'data', 'raw')

## Import

In [2]:
df = pd.read_parquet(os.path.join(DATA_FOLDER_PATH, 'raw_arrivals.parquet'))

df.head()

,access_route,state,origin_country,month,year,arrival_count
0,Aérea,Amazonas,África do Sul,janeiro,1989,9.0
1,Aérea,Amazonas,Angola,janeiro,1989,0.0
2,Aérea,Amazonas,Nigéria,janeiro,1989,0.0
3,Aérea,Amazonas,Outros países,janeiro,1989,0.0
4,Aérea,Amazonas,Costa Rica,janeiro,1989,6.0


## Consistency

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 978472 entries, 0 to 978471
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   access_route    978472 non-null  object 
 1   state           978472 non-null  object 
 2   origin_country  978472 non-null  object 
 3   month           978472 non-null  object 
 4   year            978472 non-null  int64  
 5   arrival_count   968620 non-null  float64
dtypes: float64(1), int64(1), object(4)
memory usage: 44.8+ MB


### Checking NaNs

In [4]:
df.isna().mean()

access_route      0.000000
state             0.000000
origin_country    0.000000
month             0.000000
year              0.000000
arrival_count     0.010069
dtype: float64

In [5]:
df[df['arrival_count'].isna()].head()

,access_route,state,origin_country,month,year,arrival_count
16464,Fluvial,Mato Grosso do Sul,África do Sul,janeiro,1989,NaN
16465,Fluvial,Mato Grosso do Sul,Angola,janeiro,1989,NaN
16466,Fluvial,Mato Grosso do Sul,Nigéria,janeiro,1989,NaN
16467,Fluvial,Mato Grosso do Sul,Outros países,janeiro,1989,NaN
16468,Fluvial,Mato Grosso do Sul,Costa Rica,janeiro,1989,NaN


In [6]:
df = df.dropna(subset=['arrival_count'])

## Cleaning

### Date field

In [7]:
df['year'] = df['year'].astype(str)

In [8]:
MONTH_NAME_MAPPING = {
    'janeiro': 'Janeiro',
    'fevereiro': 'Fevereiro',
    'março': 'Março',
    'marco': 'Março',
    'mar�o': 'Março',
    'abril': 'Abril',
    'maio': 'Maio',
    'junho': 'Junho',
    'julho': 'Julho',
    'agosto': 'Agosto',
    'setembro': 'Setembro',
    'outubro': 'Outubro',
    'novembro': 'Novembro',
    'dezembro': 'Dezembro',
}

MONTH_NUM_MAPPING = {
    'janeiro': '01',
    'fevereiro': '02',
    'março': '03',
    'abril': '04',
    'maio': '05',
    'junho': '06',
    'julho': '07',
    'agosto': '08',
    'setembro': '09',
    'outubro': '10',
    'novembro': '11',
    'dezembro': '12',
}

df['month'] = df['month'].str.lower().map(MONTH_NAME_MAPPING)
df['month_num'] = df['month'].str.lower().map(MONTH_NUM_MAPPING)

In [9]:
df['month'].unique()

array(['Janeiro', 'Fevereiro', 'Março', 'Abril', 'Maio', 'Junho', 'Julho',
       'Agosto', 'Setembro', 'Outubro', 'Novembro', 'Dezembro'],
      dtype=object)

In [10]:
df['month_num'].unique()

array(['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11',
       '12'], dtype=object)

In [11]:
df['date'] = pd.to_datetime(
    df['year'] + '-' + df['month_num'] + '-01'
) + pd.offsets.MonthEnd(0)

In [12]:
df['date'].isna().sum()

np.int64(0)

In [13]:
df['year_month'] = (
    pd
    .to_datetime(df['year'] + '-' + df['month_num'])
    .dt.to_period('M')
    .dt.to_timestamp('M')
)

### Access Route field

In [14]:
df['access_route'].unique()

array(['Aérea', 'Terrestre', 'Fluvial', 'Marítima', 'Aéreo', 'Marítimo',
       'A�rea', 'Mar�tima'], dtype=object)

In [15]:
ACCESS_ROUTE_MAPPING = {
    'aérea': 'Aérea',
    'aéreo': 'Aérea',
    'a�rea': 'Aérea',
    'terrestre': 'Terrestre',
    'fluvial': 'Fluvial',
    'marítima': 'Marítima',
    'marítimo': 'Marítima',
    'mar�tima': 'Marítima',
}

df['access_route'] = df['access_route'].str.lower().map(ACCESS_ROUTE_MAPPING)

In [16]:
df['access_route'].unique()

array(['Aérea', 'Terrestre', 'Fluvial', 'Marítima'], dtype=object)

### State field

In [17]:
df['state'].unique()

array(['Amazonas', 'Bahia', 'Mato Grosso do Sul', 'Pará', 'Paraná',
       'Pernambuco', 'Rio Grande do Sul', 'Rio de Janeiro', 'São Paulo',
       'Outras Unidades da Federação', 'Distrito Federal',
       'Santa Catarina', 'Ceará', 'Rio Grande do Norte', 'Minas Gerais',
       'Acre', 'Amapá', 'Roraima', 'S�o Paulo', 'Paran�', 'Cear�', 'Par�',
       'Alagoas', 'Goi�s', 'Rond�nia', 'Amap�', 'Mato Grosso', 'Para�ba',
       'Esp�rito Santo', 'Maranh�o', 'Sergipe'], dtype=object)

In [18]:
STATE_MAPPING = {
    'acre': 'Acre',
    'alagoas': 'Alagoas',
    'amapá': 'Amapá',
    'amazonas': 'Amazonas',
    'bahia': 'Bahia',
    'ceará': 'Ceará',
    'distrito federal': 'Distrito Federal',
    'mato grosso': 'Mato Grosso',
    'mato grosso do sul': 'Mato Grosso do Sul',
    'minas gerais': 'Minas Gerais',
    'outras unidades da federação': 'Outras Unidades da Federação',
    'pará': 'Pará',
    'paraná': 'Paraná',
    'pernambuco': 'Pernambuco',
    'rio de janeiro': 'Rio de Janeiro',
    'rio grande do norte': 'Rio Grande do Norte',
    'rio grande do sul': 'Rio Grande do Sul',
    'roraima': 'Roraima',
    'santa catarina': 'Santa Catarina',
    'são paulo': 'São Paulo',
    'sergipe': 'Sergipe',
    'amap�': 'Amapá',
    'cear�': 'Ceará',
    'esp�rito santo': 'Espírito Santo',
    'goi�s': 'Goiás',
    'maranh�o': 'Maranhão',
    'para�ba': 'Paraíba',
    'par�': 'Pará',
    'paran�': 'Paraná',
    'rond�nia': 'Rondônia',
    's�o paulo': 'São Paulo',
}

df['state'] = df['state'].str.lower().map(STATE_MAPPING)

In [19]:
df['state'].unique()

array(['Amazonas', 'Bahia', 'Mato Grosso do Sul', 'Pará', 'Paraná',
       'Pernambuco', 'Rio Grande do Sul', 'Rio de Janeiro', 'São Paulo',
       'Outras Unidades da Federação', 'Distrito Federal',
       'Santa Catarina', 'Ceará', 'Rio Grande do Norte', 'Minas Gerais',
       'Acre', 'Amapá', 'Roraima', 'Alagoas', 'Goiás', 'Rondônia',
       'Mato Grosso', 'Paraíba', 'Espírito Santo', 'Maranhão', 'Sergipe'],
      dtype=object)

In [20]:
STATE_REGION_MAPPING = {
    'Acre': 'Região Norte',
    'Amapá': 'Região Norte',
    'Amazonas': 'Região Norte',
    'Pará': 'Região Norte',
    'Rondônia': 'Região Norte',
    'Roraima': 'Região Norte',
    'Alagoas': 'Região Nordeste',
    'Bahia': 'Região Nordeste',
    'Ceará': 'Região Nordeste',
    'Maranhão': 'Região Nordeste',
    'Paraíba': 'Região Nordeste',
    'Pernambuco': 'Região Nordeste',
    'Rio Grande do Norte': 'Região Nordeste',
    'Sergipe': 'Região Nordeste',
    'Distrito Federal': 'Região Centro-Oeste',
    'Goiás': 'Região Centro-Oeste',
    'Mato Grosso': 'Região Centro-Oeste',
    'Mato Grosso do Sul': 'Região Centro-Oeste',
    'Espírito Santo': 'Região Sudeste',
    'Minas Gerais': 'Região Sudeste',
    'Rio de Janeiro': 'Região Sudeste',
    'São Paulo': 'Região Sudeste',
    'Paraná': 'Região Sul',
    'Rio Grande do Sul': 'Região Sul',
    'Santa Catarina': 'Região Sul',
    'Outras Unidades da Federação': 'Outros',
}

df['state_region'] = df['state'].map(STATE_REGION_MAPPING)
df['state_region'].value_counts(dropna=False)

state_region
Região Sul             276630
Região Norte           204222
Região Nordeste        180131
Região Sudeste         135251
Outros                  94831
Região Centro-Oeste     77555
Name: count, dtype: int64

### Origin Country field

In [21]:
df[df['origin_country'].str.contains('�')]['origin_country'].unique()

array(['Afeganist�o', 'Alb�nia', 'Ant�gua e Barbuda', 'Arg�lia',
       'Arm�nia', 'Ar�bia Saudita', 'Austr�lia', 'Azerbaij�o',
       'Bielor�ssia', 'Bol�via', 'Bulg�ria', 'But�o', 'B�lgica',
       'B�snia-Herzegovina', 'Camar�es', 'Canad�', 'Casaquist�o',
       'Col�mbia', 'Cro�cia', 'Emirados �rabes Unidos', 'Eritr�ia',
       'Eslov�quia', 'Eslov�nia', 'Est�nia', 'Eti�pia', 'Finl�ndia',
       'Fran�a', 'Gab�o', 'Ge�rgia', 'Gr�cia', 'Guin�',
       'Guin� Equatorial', 'Guin�-Bissau', 'G�mbia', 'Indon�sia', 'Ir�',
       'Isl�ndia', 'It�lia', 'I�men', 'Jap�o', 'Jord�nia', 'Let�nia',
       'Lib�ria', 'Litu�nia', 'L�bano', 'L�bia', 'Maced�nia', 'Malau�',
       'Mal�sia', 'Maurit�nia', 'Maur�cio', 'Mold�via', 'Mong�lia',
       'Mo�ambique', 'M�xico', 'M�naco', 'Nam�bia', 'Nicar�gua',
       'Nig�ria', 'Nova Zel�ndia', 'Om�', 'Panam�', 'Papua Nova Guin�',
       'Paquist�o', 'Pa�ses n�o especificados', 'Pol�nia', 'Quirguist�o',
       'Qu�nia', 'Rep�blica Centro Africana',
       '

In [22]:
COUNTRY_MAPPING = {
    'afeganistão': 'Afeganistão',
    'alemanha': 'Alemanha',
    'andorra': 'Andorra',
    'angola': 'Angola',
    'antilhas holandesas': 'Antilhas Holandesas',
    'arábia saudita': 'Arábia Saudita',
    'argentina': 'Argentina',
    'austrália': 'Austrália',
    'bahamas': 'Bahamas',
    'bangladesh': 'Bangladesh',
    'barbados': 'Barbados',
    'barein': 'Barein',
    'belize': 'Belize',
    'benin': 'Benin',
    'bolívia': 'Bolívia',
    'botsuana': 'Botsuana',
    'brunei': 'Brunei',
    'bulgária': 'Bulgária',
    'burkina faso': 'Burkina Faso',
    'burundi': 'Burundi',
    'bélgica': 'Bélgica',
    'cabo verde': 'Cabo Verde',
    'camboja': 'Camboja',
    'canadá': 'Canadá',
    'catar': 'Catar',
    'chade': 'Chade',
    'chile': 'Chile',
    'china': 'China',
    'china, hong kong': 'China, Hong Kong',
    'china, macao': 'China, Macao',
    'chipre': 'Chipre',
    'cingapura': 'Cingapura',
    'colômbia': 'Colômbia',
    'comores': 'Comores',
    'congo': 'Congo',
    'coreia do norte': 'Coreia do Norte',
    'costa do marfim': 'Costa do Marfim',
    'costa rica': 'Costa Rica',
    'croácia': 'Croácia',
    'cuba': 'Cuba',
    'dinamarca': 'Dinamarca',
    'djibouti': 'Djibouti',
    'dominica': 'Dominica',
    'egito': 'Egito',
    'el salvador': 'El Salvador',
    'equador': 'Equador',
    'eslováquia': 'Eslováquia',
    'eslovênia': 'Eslovênia',
    'espanha': 'Espanha',
    'estados unidos': 'Estados Unidos',
    'estônia': 'Estônia',
    'fiji': 'Fiji',
    'filipinas': 'Filipinas',
    'finlândia': 'Finlândia',
    'frança': 'França',
    'gana': 'Gana',
    'granada': 'Granada',
    'grécia': 'Grécia',
    'guatemala': 'Guatemala',
    'guiana': 'Guiana',
    'guiana francesa': 'Guiana Francesa',
    'haiti': 'Haiti',
    'holanda': 'Holanda',
    'honduras': 'Honduras',
    'hungria': 'Hungria',
    'ilhas marshall': 'Ilhas Marshall',
    'indonésia': 'Indonésia',
    'iraque': 'Iraque',
    'irlanda': 'Irlanda',
    'irã': 'Irã',
    'israel': 'Israel',
    'itália': 'Itália',
    'jamaica': 'Jamaica',
    'japão': 'Japão',
    'kiribati': 'Kiribati',
    'kuwait': 'Kuwait',
    'laos': 'Laos',
    'lesoto': 'Lesoto',
    'letônia': 'Letônia',
    'liechtenstein': 'Liechtenstein',
    'lituânia': 'Lituânia',
    'luxemburgo': 'Luxemburgo',
    'líbano': 'Líbano',
    'madagascar': 'Madagascar',
    'maldivas': 'Maldivas',
    'mali': 'Mali',
    'malta': 'Malta',
    'malásia': 'Malásia',
    'marrocos': 'Marrocos',
    'moçambique': 'Moçambique',
    'montenegro': 'Montenegro',
    'mianma': 'Mianma',
    'méxico': 'México',
    'nauru': 'Nauru',
    'nepal': 'Nepal',
    'nicarágua': 'Nicarágua',
    'nigéria': 'Nigéria',
    'noruega': 'Noruega',
    'nova zelândia': 'Nova Zelândia',
    'outros países': 'Outros países',
    'outros países da américa central e caribe': 'Outros países da América Central e Caribe',
    'outros países da europa': 'Outros países da Europa',
    'outros países da oceania': 'Outros países da Oceania',
    'outros países da ásia': 'Outros países da Ásia',
    'outros países da áfrica': 'Outros países da África',
    'palau': 'Palau',
    'palestina': 'Palestina',
    'panamá': 'Panamá',
    'papua nova guiné': 'Papua Nova Guiné',
    'paquistão': 'Paquistão',
    'paraguai': 'Paraguai',
    'países não especificados': 'Países não especificados',
    'peru': 'Peru',
    'polônia': 'Polônia',
    'porto rico': 'Porto Rico',
    'portugal': 'Portugal',
    'quirguistão': 'Quirguistão',
    'quênia': 'Quênia',
    'reino unido': 'Reino Unido',
    'república da coreia': 'República da Coreia',
    'república do kosovo': 'República do Kosovo',
    'república dominicana': 'República Dominicana',
    'república tcheca': 'República Tcheca',
    'romênia': 'Romênia',
    'ruanda': 'Ruanda',
    'rússia': 'Rússia',
    'samoa': 'Samoa',
    'san marino': 'San Marino',
    'seichelles': 'Seichelles',
    'senegal': 'Senegal',
    'serra leoa': 'Serra Leoa',
    'sri lanka': 'Sri Lanka',
    'suriname': 'Suriname',
    'suécia': 'Suécia',
    'suíça': 'Suíça',
    'sérvia': 'Sérvia',
    'síria': 'Síria',
    'tadjiquistão': 'Tadjiquistão',
    'tailândia': 'Tailândia',
    'taiwan': 'Taiwan',
    'timor leste': 'Timor Leste',
    'togo': 'Togo',
    'tonga': 'Tonga',
    'trinidad e tobago': 'Trinidad e Tobago',
    'tunísia': 'Tunísia',
    'turquia': 'Turquia',
    'tuvalu': 'Tuvalu',
    'ucrânia': 'Ucrânia',
    'uganda': 'Uganda',
    'uruguai': 'Uruguai',
    'uzbequistão': 'Uzbequistão',
    'vanuatu': 'Vanuatu',
    'vaticano': 'Vaticano',
    'venezuela': 'Venezuela',
    'áfrica do sul': 'África do Sul',
    'áustria': 'Áustria',
    'índia': 'Índia',
    'afeganist�o': 'Afeganistão',
    'alb�nia': 'Albânia',
    'ant�gua e barbuda': 'Antígua e Barbuda',
    'arg�lia': 'Argélia',
    'arm�nia': 'Armênia',
    'ar�bia saudita': 'Arábia Saudita',
    'austr�lia': 'Austrália',
    'azerbaij�o': 'Azerbaijão',
    'bielor�ssia': 'Bielorrússia',
    'bol�via': 'Bolívia',
    'bulg�ria': 'Bulgária',
    'but�o': 'Butão',
    'b�lgica': 'Bélgica',
    'b�snia-herzegovina': 'Bósnia-Herzegovina',
    'camar�es': 'Camarões',
    'canad�': 'Canadá',
    'casaquist�o': 'Casaquistão',
    'col�mbia': 'Colômbia',
    'cro�cia': 'Croácia',
    'emirados �rabes unidos': 'Emirados Árabes Unidos',
    'eritr�ia': 'Eritreia',
    'eslov�quia': 'Eslováquia',
    'eslov�nia': 'Eslovênia',
    'est�nia': 'Estônia',
    'eti�pia': 'Etiópia',
    'finl�ndia': 'Finlândia',
    'fran�a': 'França',
    'gab�o': 'Gabão',
    'ge�rgia': 'Geórgia',
    'gr�cia': 'Grécia',
    'guin�': 'Guiné',
    'guin� equatorial': 'Guiné Equatorial',
    'guin�-bissau': 'Guiné-Bissau',
    'g�mbia': 'Gâmbia',
    'ilhas salom�o': 'Ilhas Salomão',
    'indon�sia': 'Indonésia',
    'ir�': 'Irã',
    'isl�ndia': 'Islândia',
    'it�lia': 'Itália',
    'i�men': 'Iêmen',
    'jap�o': 'Japão',
    'jord�nia': 'Jordânia',
    'let�nia': 'Letônia',
    'lib�ria': 'Libéria',
    'litu�nia': 'Lituânia',
    'l�bano': 'Líbano',
    'l�bia': 'Líbia',
    'maced�nia': 'Macedônia',
    'malau�': 'Malauí',
    'mal�sia': 'Malásia',
    'maurit�nia': 'Mauritânia',
    'maur�cio': 'Maurício',
    'micron�sia': 'Micronésia',
    'mold�via': 'Moldávia',
    'mong�lia': 'Mongólia',
    'mo�ambique': 'Moçambique',
    'm�xico': 'México',
    'm�naco': 'Mônaco',
    'nam�bia': 'Namíbia',
    'nicar�gua': 'Nicarágua',
    'nig�ria': 'Nigéria',
    'nova zel�ndia': 'Nova Zelândia',
    'n�ger': 'Níger',
    'om�': 'Omã',
    'panam�': 'Panamá',
    'papua nova guin�': 'Papua Nova Guiné',
    'paquist�o': 'Paquistão',
    'pa�ses n�o especificados': 'Países não especificados',
    'pol�nia': 'Polônia',
    'quirguist�o': 'Quirguistão',
    'qu�nia': 'Quênia',
    'rep�blica centro africana': 'República Centro-Africana',
    'rep�blica democr�tica do congo': 'República Democrática do Congo',
    'rep�blica da coreia': 'República da Coreia',
    'rep�blica do kosovo': 'República do Kosovo',
    'rep�blica dominicana': 'República Dominicana',
    'rep�blica tcheca': 'República Tcheca',
    'rom�nia': 'Romênia',
    'r�ssia': 'Rússia',
    'santa l�cia': 'Santa Lúcia',
    'som�lia': 'Somália',
    'suazil�ndia': 'Suazilândia',
    'sud�o': 'Sudão',
    'sud�o do sul': 'Sudão do Sul',
    'su�cia': 'Suécia',
    'su��a': 'Suíça',
    's�o crist�v�o e nevis': 'São Cristóvão e Nevis',
    's�o tome e pr�ncipe': 'São Tomé e Príncipe',
    's�o vicente e granadinas': 'São Vicente e Granadinas',
    's�rvia': 'Sérvia',
    's�ria': 'Síria',
    'tadjiquist�o': 'Tadjiquistão',
    'tail�ndia': 'Tailândia',
    'tanz�nia': 'Tanzânia',
    'tun�sia': 'Tunísia',
    'turcomenist�o': 'Turcomenistão',
    'ucr�nia': 'Ucrânia',
    'uzbequist�o': 'Uzbequistão',
    'vietn�': 'Vietnã',
    'zimb�bue': 'Zimbábue',
    'z�mbia': 'Zâmbia',
    '�frica do sul': 'África do Sul',
    '�ndia': 'Índia',
    '�ustria': 'Áustria',
}

df['origin_country'] = df['origin_country'].str.lower().map(COUNTRY_MAPPING)

In [23]:
df[df['origin_country'].str.contains('�')]['origin_country'].unique()

array([], dtype=object)

In [24]:
COUNTRY_CULTURAL_SUBDIVISION_MAPPING = {
    # MENA
    'Afeganistão': 'MENA',
    'Argélia': 'MENA',
    'Arábia Saudita': 'MENA',
    'Barein': 'MENA',
    'Catar': 'MENA',
    'Egito': 'MENA',
    'Emirados Árabes Unidos': 'MENA',
    'Irã': 'MENA',
    'Iraque': 'MENA',
    'Israel': 'MENA',
    'Iêmen': 'MENA',
    'Jordânia': 'MENA',
    'Kuwait': 'MENA',
    'Líbano': 'MENA',
    'Líbia': 'MENA',
    'Marrocos': 'MENA',
    'Mauritânia': 'MENA',
    'Omã': 'MENA',
    'Palestina': 'MENA',
    'Síria': 'MENA',
    'Sudão': 'MENA',
    'Tunísia': 'MENA',
    'Turquia': 'MENA',
    'Djibouti': 'MENA',
    # LATAM
    'Antígua e Barbuda': 'LATAM',
    'Antilhas Holandesas': 'LATAM',
    'Argentina': 'LATAM',
    'Bahamas': 'LATAM',
    'Barbados': 'LATAM',
    'Belize': 'LATAM',
    'Bolívia': 'LATAM',
    'Chile': 'LATAM',
    'Colômbia': 'LATAM',
    'Costa Rica': 'LATAM',
    'Cuba': 'LATAM',
    'Dominica': 'LATAM',
    'El Salvador': 'LATAM',
    'Equador': 'LATAM',
    'Granada': 'LATAM',
    'Guatemala': 'LATAM',
    'Guiana': 'LATAM',
    'Guiana Francesa': 'LATAM',
    'Haiti': 'LATAM',
    'Honduras': 'LATAM',
    'Jamaica': 'LATAM',
    'México': 'LATAM',
    'Nicarágua': 'LATAM',
    'Panamá': 'LATAM',
    'Paraguai': 'LATAM',
    'Peru': 'LATAM',
    'Porto Rico': 'LATAM',
    'República Dominicana': 'LATAM',
    'Santa Lúcia': 'LATAM',
    'São Cristóvão e Nevis': 'LATAM',
    'São Vicente e Granadinas': 'LATAM',
    'Suriname': 'LATAM',
    'Trinidad e Tobago': 'LATAM',
    'Uruguai': 'LATAM',
    'Venezuela': 'LATAM',
    'Outros países da América Central e Caribe': 'LATAM',
    # APAC
    'Austrália': 'APAC',
    'Bangladesh': 'APAC',
    'Brunei': 'APAC',
    'Butão': 'APAC',
    'Camboja': 'APAC',
    'Casaquistão': 'APAC',
    'China': 'APAC',
    'China, Hong Kong': 'APAC',
    'China, Macao': 'APAC',
    'Cingapura': 'APAC',
    'Coreia do Norte': 'APAC',
    'Fiji': 'APAC',
    'Filipinas': 'APAC',
    'Ilhas Marshall': 'APAC',
    'Ilhas Salomão': 'APAC',
    'Índia': 'APAC',
    'Indonésia': 'APAC',
    'Japão': 'APAC',
    'Kiribati': 'APAC',
    'Laos': 'APAC',
    'Malásia': 'APAC',
    'Maldivas': 'APAC',
    'Micronésia': 'APAC',
    'Mianma': 'APAC',
    'Mongólia': 'APAC',
    'Nauru': 'APAC',
    'Nepal': 'APAC',
    'Nova Zelândia': 'APAC',
    'Palau': 'APAC',
    'Papua Nova Guiné': 'APAC',
    'Paquistão': 'APAC',
    'Quirguistão': 'APAC',
    'República da Coreia': 'APAC',
    'Samoa': 'APAC',
    'Sri Lanka': 'APAC',
    'Tadjiquistão': 'APAC',
    'Tailândia': 'APAC',
    'Taiwan': 'APAC',
    'Timor Leste': 'APAC',
    'Tonga': 'APAC',
    'Turcomenistão': 'APAC',
    'Tuvalu': 'APAC',
    'Uzbequistão': 'APAC',
    'Vanuatu': 'APAC',
    'Vietnã': 'APAC',
    'Outros países da Oceania': 'APAC',
    'Outros países da Ásia': 'APAC',
    # Anglo-America / English-Speaking World
    'Canadá': 'Anglo-America',
    'Estados Unidos': 'Anglo-America',
    'Irlanda': 'Anglo-America',
    'Reino Unido': 'Anglo-America',
    # Nordics / Baltics
    'Dinamarca': 'Nordics / Baltics',
    'Estônia': 'Nordics / Baltics',
    'Finlândia': 'Nordics / Baltics',
    'Islândia': 'Nordics / Baltics',
    'Letônia': 'Nordics / Baltics',
    'Lituânia': 'Nordics / Baltics',
    'Noruega': 'Nordics / Baltics',
    'Suécia': 'Nordics / Baltics',
    # Western / Eastern Europe
    'Albânia': 'Western / Eastern Europe',
    'Alemanha': 'Western / Eastern Europe',
    'Andorra': 'Western / Eastern Europe',
    'Armênia': 'Western / Eastern Europe',
    'Áustria': 'Western / Eastern Europe',
    'Azerbaijão': 'Western / Eastern Europe',
    'Bélgica': 'Western / Eastern Europe',
    'Bielorrússia': 'Western / Eastern Europe',
    'Bósnia-Herzegovina': 'Western / Eastern Europe',
    'Bulgária': 'Western / Eastern Europe',
    'Chipre': 'Western / Eastern Europe',
    'Croácia': 'Western / Eastern Europe',
    'Eslováquia': 'Western / Eastern Europe',
    'Eslovênia': 'Western / Eastern Europe',
    'Espanha': 'Western / Eastern Europe',
    'França': 'Western / Eastern Europe',
    'Geórgia': 'Western / Eastern Europe',
    'Grécia': 'Western / Eastern Europe',
    'Holanda': 'Western / Eastern Europe',
    'Hungria': 'Western / Eastern Europe',
    'Itália': 'Western / Eastern Europe',
    'Liechtenstein': 'Western / Eastern Europe',
    'Luxemburgo': 'Western / Eastern Europe',
    'Macedônia': 'Western / Eastern Europe',
    'Malta': 'Western / Eastern Europe',
    'Moldávia': 'Western / Eastern Europe',
    'Mônaco': 'Western / Eastern Europe',
    'Montenegro': 'Western / Eastern Europe',
    'Polônia': 'Western / Eastern Europe',
    'Portugal': 'Western / Eastern Europe',
    'República do Kosovo': 'Western / Eastern Europe',
    'República Tcheca': 'Western / Eastern Europe',
    'Romênia': 'Western / Eastern Europe',
    'Rússia': 'Western / Eastern Europe',
    'San Marino': 'Western / Eastern Europe',
    'Sérvia': 'Western / Eastern Europe',
    'Suíça': 'Western / Eastern Europe',
    'Ucrânia': 'Western / Eastern Europe',
    'Vaticano': 'Western / Eastern Europe',
    'Outros países da Europa': 'Western / Eastern Europe',
    # Sub-Saharan Africa
    'África do Sul': 'Sub-Saharan Africa',
    'Angola': 'Sub-Saharan Africa',
    'Benin': 'Sub-Saharan Africa',
    'Botsuana': 'Sub-Saharan Africa',
    'Burkina Faso': 'Sub-Saharan Africa',
    'Burundi': 'Sub-Saharan Africa',
    'Cabo Verde': 'Sub-Saharan Africa',
    'Camarões': 'Sub-Saharan Africa',
    'Chade': 'Sub-Saharan Africa',
    'Comores': 'Sub-Saharan Africa',
    'Congo': 'Sub-Saharan Africa',
    'Costa do Marfim': 'Sub-Saharan Africa',
    'Eritreia': 'Sub-Saharan Africa',
    'Etiópia': 'Sub-Saharan Africa',
    'Gabão': 'Sub-Saharan Africa',
    'Gana': 'Sub-Saharan Africa',
    'Guiné': 'Sub-Saharan Africa',
    'Guiné Equatorial': 'Sub-Saharan Africa',
    'Guiné-Bissau': 'Sub-Saharan Africa',
    'Gâmbia': 'Sub-Saharan Africa',
    'Lesoto': 'Sub-Saharan Africa',
    'Libéria': 'Sub-Saharan Africa',
    'Madagascar': 'Sub-Saharan Africa',
    'Malauí': 'Sub-Saharan Africa',
    'Mali': 'Sub-Saharan Africa',
    'Maurício': 'Sub-Saharan Africa',
    'Moçambique': 'Sub-Saharan Africa',
    'Namíbia': 'Sub-Saharan Africa',
    'Níger': 'Sub-Saharan Africa',
    'Nigéria': 'Sub-Saharan Africa',
    'Quênia': 'Sub-Saharan Africa',
    'República Centro-Africana': 'Sub-Saharan Africa',
    'República Democrática do Congo': 'Sub-Saharan Africa',
    'Ruanda': 'Sub-Saharan Africa',
    'São Tomé e Príncipe': 'Sub-Saharan Africa',
    'Seichelles': 'Sub-Saharan Africa',
    'Senegal': 'Sub-Saharan Africa',
    'Serra Leoa': 'Sub-Saharan Africa',
    'Somália': 'Sub-Saharan Africa',
    'Sudão do Sul': 'Sub-Saharan Africa',
    'Suazilândia': 'Sub-Saharan Africa',
    'Tanzânia': 'Sub-Saharan Africa',
    'Togo': 'Sub-Saharan Africa',
    'Uganda': 'Sub-Saharan Africa',
    'Zimbábue': 'Sub-Saharan Africa',
    'Zâmbia': 'Sub-Saharan Africa',
    'Outros países da África': 'Sub-Saharan Africa',
    # Other / unclassified
    'Outros países': 'Outros',
    'Países não especificados': 'Outros',
}

df['country_cultural_subdivision'] = df['origin_country'].map(
    COUNTRY_CULTURAL_SUBDIVISION_MAPPING
)
df['country_cultural_subdivision'].value_counts(dropna=False)

country_cultural_subdivision
LATAM                       276441
Western / Eastern Europe    226395
APAC                        124477
Outros                       88194
Nordics / Baltics            67147
Sub-Saharan Africa           66577
MENA                         64327
Anglo-America                55062
Name: count, dtype: int64

## Final Validation

In [25]:
print(f'Null counts:\n{df.isna().sum()}')
print(f'\nUnique countries: {df["origin_country"].nunique()}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')

Null counts:
access_route                    0
state                           0
origin_country                  0
month                           0
year                            0
arrival_count                   0
month_num                       0
date                            0
year_month                      0
state_region                    0
country_cultural_subdivision    0
dtype: int64

Unique countries: 208
Date range: 1989-01-31 00:00:00 to 2025-12-31 00:00:00


In [26]:
df.to_parquet(
    os.path.join(ROOT_PATH, 'data', 'processed', 'cleaned_arrivals.parquet'),
    index=False,
)